# Solar Filament V5 — pretrained Mask R-CNN + crop U-Net

Modified research reproduction of [Nomannic's Apache-2.0 Kaggle notebook](https://www.kaggle.com/code/nomannic19/cv-solar-filament-mask-r-cnn-u-net-refiner). Original author: **Nomannic**; original title: *[CV] Solar Filament | Mask R-CNN + U-Net Refiner*. Licensed Apache-2.0: https://www.apache.org/licenses/LICENSE-2.0 . See our third_party/NOTICE.md.

Changes: remove EDA-only cells, GPU preflight, four-flip refiner TTA ablation selected on official TRAIN validation, submission RLE gate. The reference's two-T4 detector training, pretrained COCO weights, ResNet18 crop refiner, and validation threshold selection are retained. No test labels or metadata are used for training. Requires Kaggle Internet On + GPU T4 x2. Estimated >1h GPU; score not yet known. Preserve the 0.24 submission until this candidate is verified.


In [ ]:
import torch, cv2, torchvision, importlib.util
assert torch.cuda.is_available(), 'Enable Kaggle GPU.'
assert torch.cuda.device_count() >= 2, 'This two-worker DDP notebook requires Kaggle T4 x2.'
assert importlib.util.find_spec('pycocotools'), 'Install pycocotools.'
from torchvision.ops import nms
assert nms(torch.tensor([[0.,0.,10.,10.]],device='cuda:0'),torch.tensor([1.],device='cuda:0'),0.5).numel()==1
print('Preflight passed: two GPUs, torchvision CUDA ops, COCO RLE.')


In [ ]:
from pathlib import Path
import json, random, pickle, cv2, numpy as np, pandas as pd, torch
from pycocotools.coco import COCO
from sklearn.model_selection import train_test_split

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_ROOT = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026")
ANN_PATH = DATA_ROOT / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
TRAIN_DIR, TEST_DIR = DATA_ROOT / "train/train_images", DATA_ROOT / "test/test_images"
WORK_DIR = Path("/kaggle/working/solar_filaments_final")
WORK_DIR.mkdir(parents=True, exist_ok=True)

coco = COCO(str(ANN_PATH))
records = []
for image_id, image_info in coco.imgs.items():
    annotations = coco.imgToAnns[image_id]
    file_name = image_info["file_name"]
    records.append({"id": str(image_id), "file_name": file_name, "observation_id": Path(file_name).stem, "site": Path(file_name).stem[-2:], "n_filaments": len(annotations), "total_mask_area": sum(float(annotation["area"]) for annotation in annotations)})

records_df = pd.DataFrame(records)
manifest = records_df.sort_values(["file_name", "n_filaments", "total_mask_area", "id"]).drop_duplicates("file_name", keep="last").reset_index(drop=True)
manifest["count_bin"] = pd.qcut(manifest["n_filaments"], 4, labels=False, duplicates="drop")
strata = manifest["site"] + "_" + manifest["count_bin"].astype(str)
train_indices, val_indices = train_test_split(manifest.index, test_size=0.20, random_state=SEED, stratify=strata)
manifest["split"] = np.where(manifest.index.isin(val_indices), "val", "train")
manifest.to_csv(WORK_DIR / "training_manifest.csv", index=False)

print(f"Selected physical observations: {len(manifest)}")
print(f"Training observations:          {(manifest.split == 'train').sum()}")
print(f"Validation observations:        {(manifest.split == 'val').sum()}")
display(pd.crosstab(manifest.site, manifest.split))

In [ ]:
%%writefile /kaggle/working/solar_filaments_final/train_detector.py
import os, argparse, random, time
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch, torch.distributed as dist
from pycocotools.coco import COCO
from pycocotools import mask as mask_utils
from scipy.optimize import linear_sum_assignment
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign
from tqdm.auto import tqdm

ROOT = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026")
ANN_PATH, IMAGE_DIR = ROOT / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json", ROOT / "train/train_images"
WORK_DIR = Path("/kaggle/working/solar_filaments_final")
MANIFEST_PATH = WORK_DIR / "training_manifest.csv"
IMAGE_SIZE, SCORE_THRESHOLD, MASK_THRESHOLD, SEED = 1024, 0.60, 0.50, 2026
cv2.setNumThreads(0)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    seed = torch.initial_seed() % 2**32
    random.seed(seed); np.random.seed(seed)

def masks_to_boxes(masks):
    boxes = []
    for mask in masks:
        ys, xs = np.where(mask)
        boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
    return np.asarray(boxes, dtype=np.float32).reshape(-1, 4)

def encode_mask(mask):
    return mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))

def merge_overlaps(masks):
    masks = [np.asarray(mask, dtype=bool) for mask in masks if np.any(mask)]
    if len(masks) < 2:
        return masks
    rles = [encode_mask(mask) for mask in masks]
    adjacency = mask_utils.iou(rles, rles, [0] * len(rles)) > 0
    groups, seen = [], set()
    for start in range(len(masks)):
        if start in seen:
            continue
        stack, group = [start], []
        while stack:
            index = stack.pop()
            if index in seen:
                continue
            seen.add(index); group.append(index)
            stack.extend(np.flatnonzero(adjacency[index]).tolist())
        groups.append(group)
    return [np.logical_or.reduce([masks[index] for index in group]) for group in groups]

def instance_metrics(pred_masks, true_masks):
    p, t = len(pred_masks), len(true_masks)
    if p and t:
        iou = mask_utils.iou([encode_mask(mask) for mask in pred_masks], [encode_mask(mask) for mask in true_masks], [0] * t)
        dice = 2 * iou / (1 + iou + 1e-7)
        rows, cols = linear_sum_assignment(-dice)
        mean_dice = float(dice[rows, cols].sum() / max(p, t))
        tp = int((iou[rows, cols] >= 0.5).sum())
    else:
        mean_dice, tp = float(p == 0 and t == 0), 0

    if p or t:
        shape = pred_masks[0].shape if p else true_masks[0].shape
        pred_union = np.logical_or.reduce(pred_masks) if p else np.zeros(shape, bool)
        true_union = np.logical_or.reduce(true_masks) if t else np.zeros(shape, bool)
        intersection = np.logical_and(pred_union, true_union).sum()
        union_dice = float(2 * intersection / (pred_union.sum() + true_union.sum() + 1e-7))
    else:
        union_dice = 1.0

    precision, recall = tp / p if p else 0.0, tp / t if t else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"mean_dice": mean_dice, "union_dice": union_dice, "f1_50": f1, "count_error": abs(p - t), "predictions": p}

class FilamentDataset(Dataset):
    def __init__(self, records, augment=False):
        self.records, self.augment = records.reset_index(drop=True), augment
        self.coco = COCO(str(ANN_PATH))

    def __len__(self):
        return len(self.records)

    @staticmethod
    def photometric(image):
        valid = image > 0.01

        if np.random.rand() < 0.70:
            image[valid] = image[valid] ** np.random.uniform(0.80, 1.25)

        if np.random.rand() < 0.70:
            mean = image[valid].mean()
            contrast, gain = np.random.uniform(0.85, 1.15), np.random.uniform(0.90, 1.10)
            image[valid] = ((image[valid] - mean) * contrast + mean) * gain

        if np.random.rand() < 0.15:
            blurred = cv2.GaussianBlur(image, (0, 0), np.random.uniform(0.30, 0.80))
            image[valid] = blurred[valid]

        if np.random.rand() < 0.25:
            image[valid] += np.random.normal(0, np.random.uniform(0.003, 0.015), valid.sum())

        return np.clip(image, 0, 1)

    def __getitem__(self, index):
        row = self.records.iloc[index]
        image_id = str(row["id"])
        image = cv2.imread(str(IMAGE_DIR / row["file_name"]), cv2.IMREAD_GRAYSCALE)
        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32) / 255

        masks = []
        for annotation in self.coco.imgToAnns[image_id]:
            mask = cv2.resize(self.coco.annToMask(annotation), (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST).astype(bool)
            if np.any(mask):
                masks.append(mask)
        masks = np.stack(masks) if masks else np.zeros((0, IMAGE_SIZE, IMAGE_SIZE), bool)

        if self.augment:
            rotations = np.random.randint(4)
            image = np.rot90(image, rotations)
            masks = np.rot90(masks, rotations, axes=(1, 2))

            if np.random.rand() < 0.5:
                image = np.flip(image, axis=1)
                masks = np.flip(masks, axis=2)

            image = self.photometric(image)

        image, masks = np.ascontiguousarray(image), np.ascontiguousarray(masks)
        boxes = masks_to_boxes(masks)
        target = {"boxes": torch.as_tensor(boxes, dtype=torch.float32), "labels": torch.ones(len(masks), dtype=torch.int64), "masks": torch.as_tensor(masks, dtype=torch.uint8), "image_id": torch.tensor([index]), "area": torch.as_tensor(masks.sum((1, 2)), dtype=torch.float32), "iscrowd": torch.zeros(len(masks), dtype=torch.int64)}
        image = torch.from_numpy(image).unsqueeze(0).repeat(3, 1, 1)
        return image, target, str(row["observation_id"])

def collate_fn(batch):
    return tuple(zip(*batch))

def build_model(pretrained=True):
    weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None
    model = maskrcnn_resnet50_fpn_v2(weights=weights, weights_backbone=None)

    box_features = model.roi_heads.box_predictor.cls_score.in_features
    mask_features = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden = model.roi_heads.mask_predictor.conv5_mask.out_channels
    model.roi_heads.box_predictor = FastRCNNPredictor(box_features, 2)
    model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_features, hidden, 2)

    model.rpn.anchor_generator = AnchorGenerator(sizes=((16,), (32,), (64,), (128,), (256,)), aspect_ratios=((0.5, 1.0, 2.0),) * 5)
    model.roi_heads.mask_roi_pool = MultiScaleRoIAlign(featmap_names=["0", "1", "2", "3"], output_size=28, sampling_ratio=2)
    model.transform.min_size, model.transform.max_size = (IMAGE_SIZE,), IMAGE_SIZE
    return model

@torch.inference_mode()
def evaluate(model, loader, device):
    model.eval()
    values = []

    for images, targets, _ in tqdm(loader, desc="Validation", leave=False):
        output = model([images[0].to(device, non_blocking=True)])[0]
        keep = output["scores"] >= SCORE_THRESHOLD
        pred_masks = (output["masks"][keep, 0] >= MASK_THRESHOLD).byte().cpu().numpy().astype(bool)
        pred_masks = merge_overlaps(pred_masks)
        true_masks = targets[0]["masks"].numpy().astype(bool)
        values.append(instance_metrics(pred_masks, true_masks))

    return pd.DataFrame(values).mean().to_dict()

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--mode", choices=["validation", "final"], default="validation")
    parser.add_argument("--epochs", type=int, default=10)
    args = parser.parse_args()

    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    seed_everything(SEED + rank)
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

    manifest = pd.read_csv(MANIFEST_PATH, dtype={"id": str})
    train_records = manifest if args.mode == "final" else manifest[manifest["split"] == "train"]
    val_records = manifest[manifest["split"] == "val"]

    train_dataset = FilamentDataset(train_records, augment=True)
    sampler = DistributedSampler(train_dataset, shuffle=True, seed=SEED)
    train_loader = DataLoader(train_dataset, batch_size=1, sampler=sampler, num_workers=2, pin_memory=True, persistent_workers=True, collate_fn=collate_fn, worker_init_fn=seed_worker)

    val_loader = None
    if rank == 0 and args.mode == "validation":
        val_dataset = FilamentDataset(val_records, augment=False)
        val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True, collate_fn=collate_fn, worker_init_fn=seed_worker)

    model = DDP(build_model(pretrained=True).to(device), device_ids=[local_rank], broadcast_buffers=False)
    optimizer = torch.optim.SGD([parameter for parameter in model.parameters() if parameter.requires_grad], lr=0.005, momentum=0.9, weight_decay=1e-4)
    milestones = sorted(set([max(1, round(args.epochs * 0.70)), max(2, round(args.epochs * 0.90))]))
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=0.1)
    scaler = torch.amp.GradScaler("cuda")
    history, best_dice = [], -1.0

    for epoch in range(1, args.epochs + 1):
        sampler.set_epoch(epoch)
        model.train()
        totals = torch.zeros(2, device=device)
        started = time.perf_counter()

        progress = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{args.epochs}", disable=rank != 0, leave=False)
        for images, targets, _ in progress:
            images = [image.to(device, non_blocking=True) for image in images]
            targets = [{key: value.to(device, non_blocking=True) for key, value in target.items()} for target in targets]
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", dtype=torch.float16):
                losses = model(images, targets)
                loss = sum(losses.values())

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            totals[0] += loss.detach()
            totals[1] += 1

        dist.all_reduce(totals)
        train_loss = (totals[0] / totals[1]).item()
        metrics = {}

        if rank == 0 and args.mode == "validation":
            metrics = evaluate(model.module, val_loader, device)

        dist.barrier(device_ids=[local_rank])
        current_lr = optimizer.param_groups[0]["lr"]

        if rank == 0:
            row = {"epoch": epoch, "train_loss": train_loss, **metrics, "lr": current_lr, "minutes": (time.perf_counter() - started) / 60}
            history.append(row)

            if args.mode == "validation" and metrics["mean_dice"] > best_dice:
                best_dice = metrics["mean_dice"]
                torch.save({"epoch": epoch, "model": model.module.state_dict()}, WORK_DIR / "best_detector.pt")

            metric_text = f" | Dice {metrics['mean_dice']:.4f} | Union {metrics['union_dice']:.4f} | F1 {metrics['f1_50']:.4f} | Count {metrics['count_error']:.2f}" if metrics else ""
            print(f"Epoch {epoch:02d}/{args.epochs} | Loss {train_loss:.4f}{metric_text} | LR {current_lr:.6f} | {row['minutes']:.1f} min", flush=True)

        scheduler.step()
        dist.barrier(device_ids=[local_rank])

    if rank == 0:
        checkpoint_name = "last_detector.pt" if args.mode == "validation" else "final_detector.pt"
        torch.save({"epoch": args.epochs, "model": model.module.state_dict()}, WORK_DIR / checkpoint_name)
        pd.DataFrame(history).to_csv(WORK_DIR / f"{args.mode}_detector_history.csv", index=False)

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

In [ ]:
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models import ResNet18_Weights

_ = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT.get_state_dict(progress=True, check_hash=True)
_ = ResNet18_Weights.DEFAULT.get_state_dict(progress=True, check_hash=True)
del _
print("Pretrained weights are cached.")

In [ ]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/solar_filaments_final/train_detector.py --mode validation --epochs 10

In [ ]:
import sys, json, pickle, importlib
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch
from pycocotools import mask as mask_utils
from scipy.optimize import linear_sum_assignment
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

sys.path.insert(0, str(WORK_DIR))
import train_detector as detector
importlib.reload(detector)

def encode_mask(mask):
    return mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))

_reference_cache = {}
def annotation_rles(record_id, size):
    key = (str(record_id), size)

    if key not in _reference_cache:
        masks = []
        for annotation in coco.imgToAnns[str(record_id)]:
            mask = coco.annToMask(annotation)

            if mask.shape != (size, size):
                mask = cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST)

            if np.any(mask):
                masks.append(encode_mask(mask))

        _reference_cache[key] = masks

    return _reference_cache[key]

def build_reference_maps(rows, size):
    selected, all_references = {}, {}
    record_ids = records_df.groupby("file_name")["id"].apply(list).to_dict()

    for row in tqdm(rows.itertuples(index=False), total=len(rows), desc=f"Building {size}px references"):
        observation_id = str(row.observation_id)
        selected[observation_id] = [annotation_rles(str(row.id), size)]
        all_references[observation_id] = [annotation_rles(str(record_id), size) for record_id in record_ids[row.file_name]]

    return selected, all_references

def score_rles(predictions, targets):
    p, t = len(predictions), len(targets)

    if p and t:
        iou = mask_utils.iou(predictions, targets, [0] * t)
        dice = 2 * iou / (1 + iou + 1e-7)
        rows, cols = linear_sum_assignment(-dice)
        matched_dice, matched_iou = dice[rows, cols], iou[rows, cols]
        mean_dice = float(matched_dice.sum() / max(p, t))
        mean_iou = float(matched_iou.sum() / max(p, t))
        tp = int((matched_iou >= 0.5).sum())
    else:
        mean_dice = mean_iou = float(p == 0 and t == 0)
        tp = 0

    if p and t:
        pred_union, target_union = mask_utils.merge(predictions), mask_utils.merge(targets)
        intersection = float(mask_utils.area(mask_utils.merge([pred_union, target_union], intersect=True)))
        union_dice = 2 * intersection / (float(mask_utils.area(pred_union)) + float(mask_utils.area(target_union)) + 1e-7)
    else:
        union_dice = float(p == 0 and t == 0)

    precision, recall = tp / p if p else 0.0, tp / t if t else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"mean_dice": mean_dice, "mean_iou": mean_iou, "union_dice": union_dice, "f1_50": f1, "count_error": abs(p - t), "predictions": p}

def evaluate_prediction_map(predictions, references):
    rows = []

    for observation_id, annotation_sets in references.items():
        reference_scores = [score_rles(predictions.get(observation_id, []), target) for target in annotation_sets]
        rows.append(pd.DataFrame(reference_scores).mean().to_dict())

    return pd.DataFrame(rows).mean().to_dict()

val_rows = manifest[manifest["split"] == "val"].reset_index(drop=True)
selected_refs_1024, all_refs_1024 = build_reference_maps(val_rows, 1024)

val_dataset = detector.FilamentDataset(val_rows, augment=False)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True, collate_fn=detector.collate_fn, worker_init_fn=detector.seed_worker)

device = torch.device("cuda:0")
detector_model = detector.build_model(pretrained=False).to(device)
checkpoint = torch.load(WORK_DIR / "best_detector.pt", map_location=device, weights_only=False)
detector_model.load_state_dict(checkpoint["model"])
detector_model.eval()

score_grid, mask_grid = [0.40, 0.50, 0.60, 0.70, 0.80], [0.40, 0.50, 0.60]
configurations = [(score, mask) for score in score_grid for mask in mask_grid]
prediction_grids = {configuration: {} for configuration in configurations}

with torch.inference_mode():
    for images, _, observation_ids in tqdm(val_loader, desc="Caching detector predictions"):
        output = detector_model([images[0].to(device, non_blocking=True)])[0]
        keep = output["scores"] >= min(score_grid)
        scores = output["scores"][keep].cpu().numpy()
        probabilities = output["masks"][keep, 0].cpu().numpy()
        observation_id = observation_ids[0]

        for mask_threshold in mask_grid:
            binary_masks = probabilities >= mask_threshold

            for score_threshold in score_grid:
                masks = detector.merge_overlaps(binary_masks[scores >= score_threshold])
                prediction_grids[(score_threshold, mask_threshold)][observation_id] = [encode_mask(mask) for mask in masks]

calibration_rows = []
for configuration in tqdm(configurations, desc="One-time calibration"):
    selected_metrics = evaluate_prediction_map(prediction_grids[configuration], selected_refs_1024)
    all_metrics = evaluate_prediction_map(prediction_grids[configuration], all_refs_1024)
    calibration_rows.append({"score_threshold": configuration[0], "mask_threshold": configuration[1], "selected_dice": selected_metrics["mean_dice"], "all_dice": all_metrics["mean_dice"], "all_union_dice": all_metrics["union_dice"], "all_f1_50": all_metrics["f1_50"], "count_error": all_metrics["count_error"], "predictions": all_metrics["predictions"]})

calibration = pd.DataFrame(calibration_rows)
calibration["selection_score"] = (calibration["selected_dice"] + calibration["all_dice"]) / 2
best_row = calibration.sort_values(["selection_score", "all_f1_50"], ascending=False).iloc[0]
best_configuration = (float(best_row["score_threshold"]), float(best_row["mask_threshold"]))
best_detector_predictions = prediction_grids[best_configuration]

calibration.to_csv(WORK_DIR / "detector_calibration.csv", index=False)
with open(WORK_DIR / "best_detector_predictions_1024.pkl", "wb") as file:
    pickle.dump(best_detector_predictions, file)
with open(WORK_DIR / "detector_thresholds.json", "w") as file:
    json.dump({"score_threshold": best_configuration[0], "mask_threshold": best_configuration[1]}, file, indent=2)

display(calibration.sort_values("selection_score", ascending=False).head(10).round(3))
print(f"Best detector thresholds: score={best_configuration[0]:.2f}, mask={best_configuration[1]:.2f}")

del detector_model
torch.cuda.empty_cache()

In [ ]:
from tqdm.auto import tqdm

def square_bounds(mask, context=1.8, minimum=96):
    height, width = mask.shape
    ys, xs = np.where(mask)
    center_x, center_y = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(np.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(center_x - side / 2)), int(round(center_y - side / 2))
    x1, y1 = x0 + side, y0 + side

    if x0 < 0:
        x1 -= x0; x0 = 0
    if y0 < 0:
        y1 -= y0; y0 = 0
    if x1 > width:
        x0 -= x1 - width; x1 = width
    if y1 > height:
        y0 -= y1 - height; y1 = height

    return int(x0), int(y0), int(x1), int(y1)

def build_crop_cache(rows, prefix):
    total = sum(len(coco.imgToAnns[str(record_id)]) for record_id in rows["id"])
    image_path, mask_path = WORK_DIR / f"{prefix}_crop_images.npy", WORK_DIR / f"{prefix}_crop_masks.npy"
    images = np.lib.format.open_memmap(image_path, mode="w+", dtype=np.uint8, shape=(total, 256, 256))
    masks = np.lib.format.open_memmap(mask_path, mode="w+", dtype=np.uint8, shape=(total, 256, 256))
    position = 0

    for row in tqdm(rows.itertuples(index=False), total=len(rows), desc=f"Caching {prefix} crops"):
        image = cv2.imread(str(TRAIN_DIR / row.file_name), cv2.IMREAD_GRAYSCALE)

        for annotation in coco.imgToAnns[str(row.id)]:
            mask = coco.annToMask(annotation).astype(np.uint8)
            x0, y0, x1, y1 = square_bounds(mask)
            images[position] = cv2.resize(image[y0:y1, x0:x1], (256, 256), interpolation=cv2.INTER_LINEAR)
            masks[position] = cv2.resize(mask[y0:y1, x0:x1], (256, 256), interpolation=cv2.INTER_NEAREST)
            position += 1

    images.flush(); masks.flush()
    return image_path, mask_path, total

train_rows = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_rows = manifest[manifest["split"] == "val"].reset_index(drop=True)

train_crop_images, train_crop_masks, train_crop_count = build_crop_cache(train_rows, "train")
val_crop_images, val_crop_masks, val_crop_count = build_crop_cache(val_rows, "val")

print(f"Training crops:   {train_crop_count:,}")
print(f"Validation crops: {val_crop_count:,}")
print(f"Cache size: {sum(path.stat().st_size for path in [train_crop_images, train_crop_masks, val_crop_images, val_crop_masks]) / 2**20:.1f} MB")

In [ ]:
import random
from torch import nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision.models import resnet18, ResNet18_Weights

class CropDataset(Dataset):
    def __init__(self, image_path, mask_path, augment=False):
        self.images, self.masks = np.load(image_path, mmap_mode="r"), np.load(mask_path, mmap_mode="r")
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = torch.from_numpy(np.asarray(self.images[index]).copy()).float() / 255
        mask = torch.from_numpy(np.asarray(self.masks[index]).copy()).float().unsqueeze(0)

        if self.augment:
            rotations = random.randrange(4)
            image, mask = torch.rot90(image, rotations, (0, 1)), torch.rot90(mask, rotations, (1, 2))

            if random.random() < 0.5:
                image, mask = torch.flip(image, (1,)), torch.flip(mask, (2,))

            if random.random() < 0.5:
                image = image ** random.uniform(0.85, 1.20)

            if random.random() < 0.5:
                image = torch.clamp(image * random.uniform(0.90, 1.10), 0, 1)

        return image.unsqueeze(0).repeat(3, 1, 1), mask

class ConvBlock(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True), nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))

class CropUNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        encoder = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.pool, self.layer1, self.layer2, self.layer3, self.layer4 = encoder.maxpool, encoder.layer1, encoder.layer2, encoder.layer3, encoder.layer4
        self.up4, self.dec4 = nn.ConvTranspose2d(512, 256, 2, 2), ConvBlock(512, 256)
        self.up3, self.dec3 = nn.ConvTranspose2d(256, 128, 2, 2), ConvBlock(256, 128)
        self.up2, self.dec2 = nn.ConvTranspose2d(128, 64, 2, 2), ConvBlock(128, 64)
        self.up1, self.dec1 = nn.ConvTranspose2d(64, 32, 2, 2), ConvBlock(96, 32)
        self.up0, self.head = nn.ConvTranspose2d(32, 16, 2, 2), nn.Conv2d(16, 1, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, image):
        image = (image - self.mean) / self.std
        x0 = self.stem(image)
        x1 = self.layer1(self.pool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        x = self.dec4(torch.cat([self.up4(x4), x3], 1))
        x = self.dec3(torch.cat([self.up3(x), x2], 1))
        x = self.dec2(torch.cat([self.up2(x), x1], 1))
        x = self.dec1(torch.cat([self.up1(x), x0], 1))
        return self.head(self.up0(x))

def soft_dice_loss(logits, targets):
    probabilities = logits.sigmoid()
    intersection = (probabilities * targets).sum((1, 2, 3))
    denominator = probabilities.sum((1, 2, 3)) + targets.sum((1, 2, 3))
    return (1 - (2 * intersection + 1) / (denominator + 1)).mean()

def train_refiner(model, train_loader, epochs, output_path, val_loader=None):
    device = torch.device("cuda:0")
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda")
    history, best_dice = [], -1.0

    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = bce_sum = dice_sum = samples = 0

        for images, masks in tqdm(train_loader, desc=f"Refiner epoch {epoch:02d}/{epochs}", leave=False):
            images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", dtype=torch.float16):
                logits = model(images)
                bce = nn.functional.binary_cross_entropy_with_logits(logits, masks)
                dice_loss = soft_dice_loss(logits, masks)
                loss = 0.5 * bce + 0.5 * dice_loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            batch = len(images)
            loss_sum += loss.item() * batch
            bce_sum += bce.item() * batch
            dice_sum += dice_loss.item() * batch
            samples += batch

        val_dice = np.nan
        if val_loader is not None:
            model.eval()
            dice_total = val_samples = 0

            with torch.inference_mode():
                for images, masks in tqdm(val_loader, desc="Refiner validation", leave=False):
                    masks = masks.to(device, non_blocking=True).bool()
                    predictions = model(images.to(device, non_blocking=True)).sigmoid() >= 0.65
                    intersection = (predictions & masks).sum((1, 2, 3)).float()
                    denominator = predictions.sum((1, 2, 3)) + masks.sum((1, 2, 3))
                    dice_total += ((2 * intersection + 1e-7) / (denominator + 1e-7)).sum().item()
                    val_samples += len(images)

            val_dice = dice_total / val_samples

            if val_dice > best_dice:
                best_dice = val_dice
                torch.save({"epoch": epoch, "model": model.state_dict()}, output_path)

        scheduler.step()
        row = {"epoch": epoch, "loss": loss_sum / samples, "bce": bce_sum / samples, "dice_loss": dice_sum / samples, "val_dice": val_dice}
        history.append(row)
        metric_text = f" | Val Dice {val_dice:.4f}" if val_loader is not None else ""
        print(f"Epoch {epoch:02d}/{epochs} | Loss {row['loss']:.4f} | BCE {row['bce']:.4f} | Dice loss {row['dice_loss']:.4f}{metric_text}")

    if val_loader is None:
        torch.save({"epoch": epochs, "model": model.state_dict()}, output_path)

    return pd.DataFrame(history)

train_crop_dataset = CropDataset(train_crop_images, train_crop_masks, augment=True)
val_crop_dataset = CropDataset(val_crop_images, val_crop_masks, augment=False)
train_crop_loader = DataLoader(train_crop_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_crop_loader = DataLoader(val_crop_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

crop_model = CropUNet(pretrained=True)
crop_history = train_refiner(crop_model, train_crop_loader, 8, WORK_DIR / "best_refiner.pt", val_crop_loader)
crop_history.to_csv(WORK_DIR / "refiner_history.csv", index=False)
display(crop_history.round(4))

del crop_model
torch.cuda.empty_cache()
# Four-view flip TTA, selected on held-out official TRAIN observations.
def predict_refiner_probabilities(model, images, use_tta=False):
    if not use_tta:
        return model(images).sigmoid()[:,0]
    logits=[]
    for flip_y,flip_x in ((False,False),(False,True),(True,False),(True,True)):
        dims=tuple(d for enabled,d in ((flip_y,-2),(flip_x,-1)) if enabled)
        transformed=torch.flip(images,dims) if dims else images
        result=model(transformed).float()
        if dims: result=torch.flip(result,dims)
        logits.append(result)
    return torch.stack(logits).mean(0).sigmoid()[:,0]


In [ ]:
from collections import defaultdict

def merge_overlapping_rles(rles):
    if len(rles) < 2:
        return rles

    adjacency = mask_utils.iou(rles, rles, [0] * len(rles)) > 0
    groups, seen = [], set()

    for start in range(len(rles)):
        if start in seen:
            continue

        stack, group = [start], []
        while stack:
            index = stack.pop()

            if index in seen:
                continue

            seen.add(index); group.append(index)
            stack.extend(np.flatnonzero(adjacency[index]).tolist())

        groups.append(group)

    return [mask_utils.merge([rles[index] for index in group]) for group in groups]

with open(WORK_DIR / "best_detector_predictions_1024.pkl", "rb") as file:
    best_detector_predictions = pickle.load(file)

selected_refs_2048, all_refs_2048 = build_reference_maps(val_rows, 2048)
file_by_observation = manifest.set_index("observation_id")["file_name"].to_dict()
baseline_predictions_2048, tasks = {}, []

for observation_id, rles in tqdm(best_detector_predictions.items(), desc="Preparing native-resolution crops"):
    image = cv2.imread(str(TRAIN_DIR / file_by_observation[observation_id]), cv2.IMREAD_GRAYSCALE)
    height, width = image.shape
    baseline_predictions_2048[observation_id] = []

    for rle in rles:
        mask = mask_utils.decode(rle).astype(np.uint8)
        full_mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST).astype(bool)
        full_rle = encode_mask(full_mask)
        baseline_predictions_2048[observation_id].append(full_rle)
        x0, y0, x1, y1 = square_bounds(full_mask)
        crop = cv2.resize(image[y0:y1, x0:x1], (256, 256), interpolation=cv2.INTER_LINEAR)
        tasks.append({"observation_id": observation_id, "bounds": (x0, y0, x1, y1), "shape": (height, width), "crop": crop, "fallback": full_rle})

refiner = CropUNet(pretrained=False).to("cuda:0")
refiner_checkpoint = torch.load(WORK_DIR / "best_refiner.pt", map_location="cuda:0", weights_only=False)
refiner.load_state_dict(refiner_checkpoint["model"])
refiner.eval()

refined_unmerged = {'plain': defaultdict(list), 'flip4': defaultdict(list)}
refiner_threshold = 0.85

with torch.inference_mode():
    for start in tqdm(range(0, len(tasks), 32), desc="Refining detector masks"):
        batch_tasks = tasks[start:start + 32]
        images = torch.from_numpy(np.stack([task["crop"] for task in batch_tasks])).float().unsqueeze(1).repeat(1, 3, 1, 1).to("cuda:0") / 255
        probs_by_mode={
            mode: predict_refiner_probabilities(refiner,images,use_tta=(mode=='flip4')).cpu().numpy()
            for mode in ('plain','flip4')
        }
        for task_index,task in enumerate(batch_tasks):
            for mode in ('plain','flip4'):
                probability=probs_by_mode[mode][task_index]
                x0,y0,x1,y1=task['bounds']
                height,width=task['shape']
                crop_mask=cv2.resize(probability,(x1-x0,y1-y0),interpolation=cv2.INTER_LINEAR)>=refiner_threshold
                if np.any(crop_mask):
                    full_mask=np.zeros((height,width),dtype=np.uint8)
                    full_mask[y0:y1,x0:x1]=crop_mask
                    refined_unmerged[mode][task['observation_id']].append(encode_mask(full_mask))
                else:
                    refined_unmerged[mode][task['observation_id']].append(task['fallback'])

refined_by_mode={
    mode:{obs:merge_overlapping_rles(predictions.get(obs,[]))
          for obs in best_detector_predictions}
    for mode,predictions in refined_unmerged.items()
}
refined_predictions_2048=refined_by_mode['plain']
comparison=[]
for name,predictions in [('Detector',baseline_predictions_2048),
                         ('Detector + refiner',refined_by_mode['plain']),
                         ('Detector + refiner + flip4',refined_by_mode['flip4'])]:
    selected_metrics = evaluate_prediction_map(predictions, selected_refs_2048)
    all_metrics = evaluate_prediction_map(predictions, all_refs_2048)
    comparison.append({"strategy": name, "selected_dice": selected_metrics["mean_dice"], "all_dice": all_metrics["mean_dice"], "all_union_dice": all_metrics["union_dice"], "all_f1_50": all_metrics["f1_50"], "count_error": all_metrics["count_error"], "predictions": all_metrics["predictions"]})

comparison = pd.DataFrame(comparison).set_index("strategy")
with open(WORK_DIR / "refined_validation_predictions_2048.pkl", "wb") as file:
    pickle.dump(refined_predictions_2048, file)

display(comparison.round(3))

del refiner
torch.cuda.empty_cache()

In [ ]:
detector_history = pd.read_csv(WORK_DIR / "validation_detector_history.csv")
best_detector_epoch = max(6, int(detector_history.loc[detector_history["mean_dice"].idxmax(), "epoch"]))
print(f"Final detector epochs: {best_detector_epoch}")

In [ ]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/solar_filaments_final/train_detector.py --mode final --epochs {best_detector_epoch}

In [ ]:
best_refiner_epoch = max(5, int(crop_history.loc[crop_history["val_dice"].idxmax(), "epoch"]))
best_strategy=str(comparison['all_dice'].idxmax())
use_refiner=best_strategy!='Detector'
use_refiner_tta=best_strategy=='Detector + refiner + flip4'
print(f"Use refiner: {use_refiner} | Final refiner epochs: {best_refiner_epoch}")

if use_refiner:
    final_train_crops = ConcatDataset([CropDataset(train_crop_images, train_crop_masks, augment=True), CropDataset(val_crop_images, val_crop_masks, augment=True)])
    final_crop_loader = DataLoader(final_train_crops, batch_size=16, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
    final_refiner = CropUNet(pretrained=True)
    final_refiner_history = train_refiner(final_refiner, final_crop_loader, best_refiner_epoch, WORK_DIR / "final_refiner.pt")
    final_refiner_history.to_csv(WORK_DIR / "final_refiner_history.csv", index=False)
    del final_refiner
    torch.cuda.empty_cache()

with open(WORK_DIR / "detector_thresholds.json") as file:
    detector_thresholds = json.load(file)

final_configuration = {"detector_epochs": best_detector_epoch, "score_threshold": detector_thresholds["score_threshold"], "mask_threshold": detector_thresholds["mask_threshold"], "overlap_merge": True, "use_refiner": use_refiner, "refiner_flip4": use_refiner_tta, "refiner_epochs": best_refiner_epoch if use_refiner else None, "refiner_threshold": 0.85 if use_refiner else None}

with open(WORK_DIR / "final_configuration.json", "w") as file:
    json.dump(final_configuration, file, indent=2)

print(final_configuration)

# 4. Submission

In [ ]:
import json, pickle
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch
from pycocotools import mask as mask_utils
from tqdm.auto import tqdm

with open(WORK_DIR / "final_configuration.json") as file:
    final_configuration = json.load(file)

device = torch.device("cuda:0")
final_detector = detector.build_model(pretrained=False).to(device)
detector_checkpoint = torch.load(WORK_DIR / "final_detector.pt", map_location=device, weights_only=False)
final_detector.load_state_dict(detector_checkpoint["model"])
final_detector.eval()

final_refiner = None
if final_configuration["use_refiner"]:
    final_refiner = CropUNet(pretrained=False).to(device)
    refiner_checkpoint = torch.load(WORK_DIR / "final_refiner.pt", map_location=device, weights_only=False)
    final_refiner.load_state_dict(refiner_checkpoint["model"])
    final_refiner.eval()

score_threshold = final_configuration["score_threshold"]
mask_threshold = final_configuration["mask_threshold"]
refiner_threshold = final_configuration.get("refiner_threshold", 0.85)
test_files = sorted(path for path in TEST_DIR.iterdir() if path.suffix.lower() in {".jpeg", ".jpg", ".png"})
test_predictions = {}

with torch.inference_mode():
    for image_path in tqdm(test_files, desc="Predicting test images"):
        observation_id = image_path.stem
        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        height, width = image.shape
        resized = cv2.resize(image, (1024, 1024), interpolation=cv2.INTER_AREA).astype(np.float32) / 255
        tensor = torch.from_numpy(resized).unsqueeze(0).repeat(3, 1, 1).to(device)

        output = final_detector([tensor])[0]
        keep = output["scores"] >= score_threshold
        detector_masks = (output["masks"][keep, 0] >= mask_threshold).byte().cpu().numpy().astype(bool)
        detector_masks = detector.merge_overlaps(detector_masks)

        full_detector_masks, crops, bounds = [], [], []
        for mask in detector_masks:
            full_mask = cv2.resize(mask.astype(np.uint8), (width, height), interpolation=cv2.INTER_NEAREST).astype(bool)
            x0, y0, x1, y1 = square_bounds(full_mask)
            full_detector_masks.append(full_mask)
            bounds.append((x0, y0, x1, y1))
            crops.append(cv2.resize(image[y0:y1, x0:x1], (256, 256), interpolation=cv2.INTER_LINEAR))

        prediction_rles = []
        if final_refiner is not None and crops:
            crop_tensor = torch.from_numpy(np.stack(crops)).float().unsqueeze(1).repeat(1, 3, 1, 1).to(device) / 255
            probabilities = predict_refiner_probabilities(
                final_refiner,crop_tensor,
                use_tta=final_configuration.get('refiner_flip4',False)
            ).cpu().numpy()

            for full_detector_mask, probability, (x0, y0, x1, y1) in zip(full_detector_masks, probabilities, bounds):
                crop_mask = cv2.resize(probability, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR) >= refiner_threshold

                if np.any(crop_mask):
                    full_mask = np.zeros((height, width), dtype=np.uint8)
                    full_mask[y0:y1, x0:x1] = crop_mask
                    prediction_rles.append(encode_mask(full_mask))
                else:
                    prediction_rles.append(encode_mask(full_detector_mask))
        else:
            prediction_rles = [encode_mask(mask) for mask in full_detector_masks]

        prediction_rles = merge_overlapping_rles(prediction_rles)
        prediction_rles.sort(key=lambda rle: (float(mask_utils.toBbox(rle)[1]), float(mask_utils.toBbox(rle)[0])))
        test_predictions[observation_id] = prediction_rles

with open(WORK_DIR / "test_predictions_2048.pkl", "wb") as file:
    pickle.dump(test_predictions, file)

del final_detector, final_refiner
torch.cuda.empty_cache()

In [ ]:
submission_rows = []
for observation_id, rles in test_predictions.items():
    for filament_number, rle in enumerate(rles, 1):
        counts = rle["counts"].decode("ascii") if isinstance(rle["counts"], bytes) else rle["counts"]
        submission_rows.append({"filament_id": f"{observation_id}_{filament_number}", "segmentation_rle": counts})

submission = pd.DataFrame(submission_rows, columns=["filament_id", "segmentation_rle"])
submission_path = Path("/kaggle/working/submission.csv")
submission.to_csv(submission_path, index=False)

prediction_counts = pd.Series({observation_id: len(rles) for observation_id, rles in test_predictions.items()}, name="predicted_filaments")
areas = np.array([float(mask_utils.area(rle)) for rles in test_predictions.values() for rle in rles])
test_ids = {path.stem for path in test_files}

print(f"Submission:               {submission_path}")
print(f"Submission rows:          {len(submission):,}")
print(f"Test images represented:  {len(test_predictions)} / {len(test_files)}")
print(f"Unique filament IDs:      {submission.filament_id.is_unique}")
print(f"Prediction count/image:   mean={prediction_counts.mean():.2f}, median={prediction_counts.median():.1f}, min={prediction_counts.min()}, max={prediction_counts.max()}")
print(f"Predicted mask area:       median={np.median(areas):.0f}, min={areas.min():.0f}, max={areas.max():.0f}")
print(f"Images with no prediction: {(prediction_counts == 0).sum()}")
print(f"All test IDs covered:      {set(test_predictions) == test_ids}")
print(f"File size:                 {submission_path.stat().st_size / 2**20:.2f} MB")
display(submission.head())

## Submission gate
Validate 180 images and RLE integrity; test labels are never read.

In [ ]:
# Submission integrity; test images remain inference-only.
from pycocotools import mask as mask_utils
assert len(test_files)==180, f'Expected 180 test images, found {len(test_files)}'
assert set(test_predictions)=={p.stem for p in test_files}
assert list(submission.columns)==['filament_id','segmentation_rle']
assert submission.filament_id.is_unique and submission.segmentation_rle.notna().all()
assert len(submission)>0, 'Empty submission rejected'
ids=set(submission.filament_id)
for observation,masks in test_predictions.items():
    for j,rle in enumerate(masks,1):
        assert tuple(rle['size'])==(2048,2048)
        assert int(mask_utils.area(rle))>0
        assert f'{observation}_{j}' in ids
print('Submission gate passed:',len(test_files),'images,',len(submission),'RLE instances.')
print('Official public score is unknown until submission.')
